In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [3]:
#Step 1 - load the data
DATA_1 = Path("../results")
DATA_2 = Path("../../R_scripts_dea_gsea/results/annotation")

norm_counts = pd.read_csv(DATA_1/"tcga_logcpm_01.tsv", sep="\t", index_col=0)
emt_genes = pd.read_csv(DATA_2/"emt_symbol_to_ensembl.tsv", sep="\t", index_col=0)

print("Normalised counts:", norm_counts.shape)
print("EMT genes:", emt_genes.shape)

Normalised counts: (63856, 304)
EMT genes: (200, 1)


In [5]:
emt_genes.head

<bound method NDFrame.head of                 ENSEMBL
SYMBOL                 
ABI3BP  ENSG00000154175
ACTA2   ENSG00000107796
ADAM12  ENSG00000148848
ANPEP   ENSG00000166825
APLP1   ENSG00000105290
...                 ...
VEGFA   ENSG00000112715
VEGFC   ENSG00000150630
VIM     ENSG00000026025
WIPF1   ENSG00000115935
WNT5A   ENSG00000114251

[200 rows x 1 columns]>

In [6]:
#Step 2 - extract Ensembl IDs from EMT map
emt_ens = emt_genes["ENSEMBL"].astype(str).unique()
print("EMT Ensembl IDs:", len(emt_ens))

EMT Ensembl IDs: 200


In [7]:
#Step 3 - subset normalised matrix to EMT genes
emt_ens_in_counts = norm_counts.index.intersection(emt_ens)
print("EMT genes in normalised counts:", len(emt_ens_in_counts))

EMT genes in normalised counts: 200


In [8]:
emt_expr = norm_counts.loc[emt_ens_in_counts]
print("EMT expression matrix:", emt_expr.shape)

EMT expression matrix: (200, 304)


In [9]:
#Step 4 - calculate z-score for each gene across samples (row wise!)
gene_mean = emt_expr.mean(axis=1)
gene_sd = emt_expr.std(axis=1, ddof=0)

nonzero = gene_sd > 0
print("Genes with zero SD (dropped):", (~nonzero).sum())

emt_z = emt_expr.loc[nonzero].sub(gene_mean[nonzero], axis= 0).div(gene_sd[nonzero], axis=0)
print("Z-scored EMT matrix:", emt_z.shape)

Genes with zero SD (dropped): 0
Z-scored EMT matrix: (200, 304)


In [10]:
# Each gene should have mean ~0 and sd ~1 across samples
print("Mean of gene means (should be ~0):", emt_z.mean(axis=1).mean())
print("Mean of gene SDs   (should be ~1):", emt_z.std(axis=1, ddof=0).mean())


Mean of gene means (should be ~0): -1.9950591414767788e-16
Mean of gene SDs   (should be ~1): 1.0


In [11]:
emt_score = emt_z.mean(axis=0)
emt_score.name = "EMT_score"

print(emt_score.describe())
print(emt_score.head())


count    3.040000e+02
mean    -2.074364e-16
std      4.509265e-01
min     -1.187224e+00
25%     -3.407119e-01
50%      1.282270e-03
75%      2.941989e-01
max      1.414209e+00
Name: EMT_score, dtype: float64
5c8c8a6e-40d4-4f82-ac3d-90cfdee15c0a    0.443655
d9096909-1439-462d-b69c-cad1bf4f420c   -0.077657
29c17355-d646-48da-9e54-7b6dd85dd610    0.766772
29bcba51-2580-473d-9cf7-bedbd0dbad1b    0.234863
ad97b334-e034-4d33-a4d5-48c32d5d521c    0.193496
Name: EMT_score, dtype: float64


In [ ]:
#Step 5 - save EMT scored samples to TSV
out = DATA_1/"emt_scored_samples.tsv"
emt_score.to_csv(out, sep="\t")
print("Wrote:", out)

Wrote: ../results/emt_scored_samples.tsv
